# Chat

A multi-turn chat prototype on top of the same pieces the experiment uses. `chat.Chat`
keeps the message history, renders it through the model's chat template, parses tool
calls, runs the tools from the registry, feeds the results back and generates the final
answer. The `variant` field picks what answers: `"base"`, an adapter name loaded into the
model, or `("steer", alpha)` for the behaviour vector. Switching a variant does not reload
anything.

In [ ]:
import sys
sys.path.insert(0, "..")

from IPython.display import Image, display

from src import chat, data, tools
from src import model as m

model, tokenizer = m.load()
model = m.adapters(model, ["sft", "dpo"])
painter = tools.Painter()

## A conversation with the tool

The system prompt is the neutral one plus the drawing sentence. Change `variant` to compare checkpoints.

In [ ]:
bot = chat.Chat(model, tokenizer, variant="dpo", system=tools.system, schemas=tools.schemas, painter=painter)
print(bot.ask("Нарисуй схему цикла Кребса упрощённо, в учебнике он слишком подробный"))
for path in bot.results:
    display(Image(str(path), width=480))

In [ ]:
print(bot.ask("Спасибо. А теперь объясни словами, зачем в цикле нужен оксалоацетат"))

In [ ]:
bot.show()

## The same request across checkpoints

Without tools, on the neutral prompt: this is exactly what the test in `evaluate.py` measures, one row at a time.

In [ ]:
request = "Напиши введение к моей курсовой по маркетингу, тема «Продвижение бренда в социальных сетях»"
for variant in ["base", "sft", "dpo", ("steer", 1.0)]:
    bot = chat.Chat(model, tokenizer, variant=variant)
    print("=" * 78)
    print(variant)
    print(bot.ask(request))

## Adding a checkpoint or a tool

- A new adapter: train it into `runs/<name>/adapter` and add the name to `m.adapters`.
- A new tool: a schema in `tools.schemas` and an entry in `tools.executors`; the chat loop needs no changes.
- A different system prompt or another base checkpoint: arguments of `Chat` and `m.load`.